# Laboratório — Classificação de imagens de satélite com CNN (EuroSAT)

**Objetivo:** desenvolver e avaliar uma rede neural convolucional (CNN) própria para classificar
**imagens coloridas (RGB) de satélite** em **10 classes de uso e cobertura do solo**, justificando de forma
crítica a escolha do dataset.

**Roteiro (e onde cada item da atividade é atendido):**

| Etapa | Seção |
| --- | --- |
| Escolha e justificativa do dataset (≥ 4 classes, imagens coloridas) | 2 e 3 |
| Divisão em treino / validação / teste, com proporções | 4 |
| Pré-processamento (redimensionamento, normalização, data augmentation) | 5 |
| Arquitetura CNN própria (convolução, pooling, BatchNorm, Dropout, densas) | 6 |
| Hiperparâmetros (otimizador, taxa de aprendizado, perda, lote, épocas) | 7 |
| Gráficos de acurácia e erro (treino × validação) | 8 |
| Precision, Recall e F1-Score no teste + matriz de confusão | 9 |
| Imagens de teste: predição × real | 10 |
| Comparação de topologias (nº de filtros e de camadas) | 11 |
| Conclusões | 12 |

> 💡 **Dica (Colab):** ative a GPU em *Ambiente de execução → Alterar o tipo de ambiente de execução → GPU*.

## 1. Importações

Usamos apenas a API `tf.keras`. Fixamos a **semente** (Python, NumPy e TensorFlow de uma vez) para que
os resultados sejam reproduzíveis: sem isso, a inicialização dos pesos muda a cada execução.

In [ ]:
import time
import pathlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import set_random_seed

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix, ConfusionMatrixDisplay,
                             precision_recall_fscore_support, accuracy_score)

SEED = 42
set_random_seed(SEED)

print("TensorFlow:", tf.__version__)
print("GPU disponivel:", tf.config.list_physical_devices("GPU"))

## 2. Escolha do dataset — justificativa crítica

Escolhi o **EuroSAT (versão RGB)**: **27.000 imagens** de satélite (missão **Sentinel-2**, do programa europeu
Copernicus) de **64 × 64 pixels**, rotuladas em **10 classes** de uso e cobertura do solo:
*AnnualCrop* (lavoura anual), *Forest* (floresta), *HerbaceousVegetation* (vegetação herbácea), *Highway* (rodovia),
*Industrial* (área industrial), *Pasture* (pasto), *PermanentCrop* (cultivo permanente), *Residential* (área residencial),
*River* (rio) e *SeaLake* (mar/lago).

| Critério | Avaliação |
| --- | --- |
| **Complexidade** | Adequada. São imagens **coloridas (RGB)** de baixa resolução, e várias classes são visualmente parecidas (*Highway* × *River*; *AnnualCrop* × *PermanentCrop* × *Pasture* × *HerbaceousVegetation*). É bem mais difícil que o MNIST, mas o tamanho pequeno (64×64) e as ~27 mil imagens permitem treinar uma CNN do zero em poucos minutos na GPU. |
| **Originalidade** | MNIST, Fashion-MNIST e CIFAR-10 são os exemplos clássicos de aula, e fotos de objetos ou animais são os temas mais comuns. Imagem de **sensoriamento remoto** é um domínio diferente, em que a visão é de cima, sem "frente" nem "fundo" e com texturas muito diferentes das fotos do dia a dia. |
| **Confiança na fonte** | Publicado pelos autores do dataset (Helber et al., 2019, *IEEE Journal of Selected Topics in Applied Earth Observations and Remote Sensing*), muito usado como *benchmark* de classificação de uso do solo, e hospedado no Zenodo sob **licença MIT**. As imagens vêm de uma missão de observação da Terra de uma agência pública, e não de coleta aleatória na web. **Ressalva:** os rótulos foram atribuídos pelos autores, e como cada imagem é um recorte pequeno, algumas contêm mais de um tipo de cobertura. |
| **Motivação** | Classificar o uso do solo automaticamente serve para **monitoramento ambiental** (desmatamento, avanço da agricultura), **planejamento urbano** e gestão de recursos hídricos, em larga escala e sem depender de levantamento manual. |
| **Balanceamento** | **Moderado**: as classes têm entre cerca de 2.000 e 3.000 imagens (a razão entre a maior e a menor deve ficar perto de 1,5; o gráfico da próxima seção mostra os números reais). Não é crítico, mas por isso usamos divisão **estratificada** e olhamos **Precision/Recall/F1 por classe** e a média *macro*, não só a acurácia. |
| **Limitações** | As imagens cobrem regiões da **Europa**, então o modelo pode **não generalizar** para outros contextos (por exemplo, vegetação, cultivos e construções típicos do Brasil). A resolução de 64×64 limita os detalhes visíveis, e a versão RGB não usa as bandas infravermelhas, que ajudariam a separar vegetação. |

## 3. Carregar e conhecer os dados

Baixamos o arquivo `EuroSAT_RGB.zip` (cerca de 95 MB) do Zenodo e extraímos. Cada **subpasta** é uma classe.
Em vez de carregar todas as imagens na memória de uma vez, guardamos só os **caminhos** dos arquivos e os
**rótulos** (números inteiros de 0 a 9). Isso nos permite fazer a divisão treino/validação/teste de forma
estratificada antes de abrir as imagens.

O código descobre as classes pelo **nome da pasta** de cada imagem, então não depende da estrutura exata do zip.

> Se o download falhar (o Zenodo às vezes fica instável), tente rodar a célula de novo mais tarde, ou baixe o
> zip pelo navegador na [página do dataset](https://zenodo.org/records/7711810), envie para o Colab e ajuste `base_dir`.

In [ ]:
URL = "https://zenodo.org/records/7711810/files/EuroSAT_RGB.zip?download=1"
archive = tf.keras.utils.get_file(fname="EuroSAT_RGB.zip", origin=URL, extract=True, cache_subdir="eurosat")
base_dir = pathlib.Path(archive).parent

# Acha todas as imagens; a pasta de cada uma e o nome da classe
arquivos = sorted(p for p in base_dir.rglob("*") if p.suffix.lower() in (".jpg", ".jpeg"))
nomes = [p.parent.name for p in arquivos]
class_names = sorted(set(nomes))
num_classes = len(class_names)
indice = {nome: i for i, nome in enumerate(class_names)}

paths = np.array([str(p) for p in arquivos])
labels = np.array([indice[n] for n in nomes])

print("Classes:", class_names)
print("Total de imagens:", len(paths))

In [ ]:
# Balanceamento das classes
contagem = pd.Series(np.bincount(labels), index=class_names)
print(contagem)
print(f"\nRazao maior/menor classe: {contagem.max() / contagem.min():.2f}")

plt.figure(figsize=(9, 4))
barras = plt.bar(class_names, contagem.values, color="tab:blue")
plt.bar_label(barras)
plt.xticks(rotation=45, ha="right")
plt.title("Numero de imagens por classe")
plt.ylabel("Imagens")
plt.tight_layout(); plt.show()

In [ ]:
# Uma imagem de cada classe (64x64, RGB)
colunas = (num_classes + 1) // 2
plt.figure(figsize=(2.6 * colunas, 6))
for i, cls in enumerate(class_names):
    img = plt.imread(paths[np.where(labels == i)[0][0]])
    plt.subplot(2, colunas, i + 1)
    plt.imshow(img)
    plt.title(f"{cls}\n{img.shape[1]}x{img.shape[0]}", fontsize=10)
    plt.axis("off")
plt.tight_layout(); plt.show()

## 4. Divisão em treino, validação e teste

Usamos a proporção **70% treino / 15% validação / 15% teste**, de forma **estratificada**
(`stratify=labels`): cada conjunto mantém a mesma proporção de cada classe do dataset original.

- **Treino:** a rede ajusta os pesos com estes dados.
- **Validação:** usada *durante* o treino para acompanhar o overfitting, ajustar a taxa de aprendizado
  e decidir quando parar. Também é a base para comparar topologias.
- **Teste:** fica **intocado** até a avaliação final. Só ele dá uma estimativa honesta do desempenho em imagens novas.

In [ ]:
# 1) separa 70% para treino e 30% para "resto"
paths_train, paths_tmp, y_train, y_tmp = train_test_split(
    paths, labels, test_size=0.30, stratify=labels, random_state=SEED)

# 2) divide o "resto" ao meio: 15% validacao e 15% teste
paths_val, paths_test, y_val, y_test = train_test_split(
    paths_tmp, y_tmp, test_size=0.50, stratify=y_tmp, random_state=SEED)

total = len(paths)
for nome, y in [("Treino", y_train), ("Validacao", y_val), ("Teste", y_test)]:
    print(f"{nome:10s}: {len(y):5d} imagens ({len(y) / total * 100:.1f}%)")
print("\nImagens por classe (treino):", np.bincount(y_train))
print("Imagens por classe (teste): ", np.bincount(y_test))

## 5. Pré-processamento

1. **Redimensionamento:** a CNN precisa de entradas do mesmo tamanho. As imagens do EuroSAT já têm
   **64 × 64 × 3** (3 canais = RGB), então o redimensionamento é um passo de segurança que não altera nada aqui.
   Manter esse passo no pipeline deixa o código reutilizável para outros datasets.
2. **Normalização:** os pixels vão de 0 a 255. Fazemos a divisão por 255 **dentro do modelo**, com uma camada
   `Rescaling(1/255)`. Assim a mesma escala vale no treino, na validação, no teste e em imagens novas.
3. **Data augmentation** (apenas no treino): giros **horizontais e verticais**, pequenas rotações e zoom
   aleatórios. Em imagens de satélite, não existe "lado de cima" nem "lado de baixo" (uma floresta é uma floresta
   de ponta-cabeça), então os giros verticais são variações realistas. Essas camadas ficam **inativas** na
   validação, no teste e na predição.
4. **Pipeline `tf.data`:** lê os arquivos, decodifica o JPEG, embaralha só o treino, agrupa em lotes e usa
   `prefetch` para a GPU não esperar o disco. O `cache` (opcional) guarda as imagens decodificadas como `uint8`
   na memória (cerca de 330 MB para o dataset todo); se faltar memória, ponha `USAR_CACHE = False`.

In [ ]:
IMG_SIZE = 64
BATCH_SIZE = 64
USAR_CACHE = True      # coloque False se faltar memoria (o treino fica um pouco mais lento)
AUTOTUNE = tf.data.AUTOTUNE

def carregar_imagem(caminho, rotulo):
    img = tf.io.read_file(caminho)
    img = tf.image.decode_jpeg(img, channels=3)          # garante 3 canais (RGB)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))     # redimensiona (aqui ja sao 64x64)
    img = tf.cast(img, tf.uint8)                         # guarda como uint8 (0-255) para poupar memoria
    return img, rotulo

def criar_dataset(caminhos, rotulos, treino=False):
    ds = tf.data.Dataset.from_tensor_slices((caminhos, rotulos))
    ds = ds.map(carregar_imagem, num_parallel_calls=AUTOTUNE)
    if USAR_CACHE:
        ds = ds.cache()                                  # le do disco so na 1a epoca
    if treino:
        ds = ds.shuffle(2000, seed=SEED)                 # embaralha a cada epoca (so no treino)
    return ds.batch(BATCH_SIZE).prefetch(AUTOTUNE)

train_ds = criar_dataset(paths_train, y_train, treino=True)
val_ds   = criar_dataset(paths_val,   y_val)
test_ds  = criar_dataset(paths_test,  y_test)             # sem shuffle: a ordem bate com y_test

In [ ]:
# Camadas de data augmentation (ativas somente quando training=True)
data_augmentation = models.Sequential([
    layers.RandomFlip("horizontal_and_vertical"),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.1),
], name="data_augmentation")

# Visualizacao: a mesma imagem de treino com 7 variacoes aleatorias
img_exemplo, _ = next(iter(train_ds.unbatch().take(1)))
plt.figure(figsize=(12, 6))
plt.subplot(2, 4, 1)
plt.imshow(img_exemplo.numpy()); plt.title("original"); plt.axis("off")
for i in range(7):
    aug = data_augmentation(tf.expand_dims(tf.cast(img_exemplo, tf.float32), 0), training=True)
    plt.subplot(2, 4, i + 2)
    plt.imshow(aug[0].numpy().astype("uint8")); plt.title(f"aumentada {i + 1}"); plt.axis("off")
plt.tight_layout(); plt.show()

## 6. Arquitetura da CNN

A rede é montada por uma função `build_cnn(...)`, que recebe o **número de filtros de cada bloco**, o tamanho da
camada densa, a taxa de dropout e se usa BatchNorm. Assim criamos facilmente várias topologias na seção 11,
e cada chamada gera pesos **novos** (comparação justa).

**Arquitetura-base** (`filters=(32, 64, 128)`):

| Camada | Papel |
| --- | --- |
| `Input(64,64,3)` | Entrada: imagem RGB de satélite. |
| `Rescaling(1/255)` | Normaliza os pixels para 0–1. |
| `data_augmentation` | Giros, rotação e zoom aleatórios (só no treino). |
| **Bloco convolucional** × 3: `Conv2D(f, 3×3, padding="same")` → `BatchNormalization` → `ReLU` → `MaxPooling2D(2×2)` | A **convolução** aprende filtros que detectam características (bordas, cores, texturas); com mais blocos, as características ficam mais abstratas (de bordas e manchas de cor a padrões como o traçado de uma rodovia ou a textura de uma lavoura). A **BatchNorm** normaliza as ativações de cada lote, o que estabiliza e acelera o treino. A **ReLU** adiciona não linearidade. O **pooling** reduz a resolução pela metade (64 → 32 → 16 → 8) e dá tolerância a pequenos deslocamentos. O número de filtros dobra a cada bloco (32 → 64 → 128): a resolução cai, então podemos aumentar a "largura" da rede. |
| `GlobalAveragePooling2D` | Faz a média de cada mapa de características, gerando um vetor com um valor por filtro (128). Substitui o `Flatten` e tem **muito menos parâmetros** (um `Flatten` aqui geraria 8×8×128 = 8.192 valores, o que favoreceria o overfitting). |
| `Dropout(0.4)` | Desliga 40% dos neurônios aleatoriamente no treino (**regularização**). |
| `Dense(128, ReLU)` | Camada totalmente conectada que combina as características. |
| `Dropout(0.4)` | Mais regularização antes da saída. |
| `Dense(10, softmax)` | **Saída**: uma probabilidade por classe (somam 1). |

> Com `padding="same"`, a convolução mantém a resolução, e só o pooling a reduz. Isso difere do `padding="valid"` visto no MNIST (28 → 26).

In [ ]:
def build_cnn(filters=(32, 64, 128), dense_units=128, dropout=0.4,
              batch_norm=True, augmentation=True, name="cnn"):
    entrada = layers.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x = layers.Rescaling(1.0 / 255)(entrada)             # 0-255 -> 0-1
    if augmentation:
        x = data_augmentation(x)                         # so atua no treino

    # Um bloco convolucional por valor em 'filters'
    for f in filters:
        x = layers.Conv2D(f, (3, 3), padding="same", use_bias=not batch_norm)(x)
        if batch_norm:
            x = layers.BatchNormalization()(x)
        x = layers.Activation("relu")(x)
        x = layers.MaxPooling2D((2, 2))(x)

    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(dropout)(x)
    x = layers.Dense(dense_units, activation="relu")(x)
    x = layers.Dropout(dropout)(x)
    saida = layers.Dense(num_classes, activation="softmax")(x)
    return models.Model(entrada, saida, name=name)

set_random_seed(SEED)
model = build_cnn(name="cnn_base")
model.summary()

## 7. Hiperparâmetros e treinamento

| Hiperparâmetro | Valor | Observação |
| --- | --- | --- |
| **Otimizador** | Adam | Taxa de aprendizado adaptativa por parâmetro; funciona bem sem muito ajuste. |
| **Taxa de aprendizado** | 0,001 (inicial) | O `ReduceLROnPlateau` reduz pela metade se a `val_loss` não melhorar por 3 épocas (mínimo 1e-5). |
| **Função de perda** | `sparse_categorical_crossentropy` | Equivale à *categorical crossentropy* da aula, mas aceita rótulos inteiros (0–9) em vez de one-hot. |
| **Tamanho do lote** | 64 | Imagens processadas por passo de atualização dos pesos. |
| **Épocas** | até 30 | Com `EarlyStopping` (paciência 6 sobre `val_loss`, restaurando os melhores pesos), então o treino pode terminar antes. |
| **Regularização** | Dropout 0,4 · BatchNorm · data augmentation · EarlyStopping | Combate o overfitting. |
| **Métrica acompanhada** | acurácia | Na avaliação final usamos também Precision, Recall e F1. |

In [ ]:
EPOCHS = 30
LEARNING_RATE = 1e-3

def compilar(modelo, lr=LEARNING_RATE):
    modelo.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )

def treinar(modelo, epocas, verbose=1):
    callbacks = [
        EarlyStopping(monitor="val_loss", patience=6, restore_best_weights=True),
        ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-5),
    ]
    return modelo.fit(train_ds, validation_data=val_ds, epochs=epocas,
                      callbacks=callbacks, verbose=verbose)

compilar(model)
history = treinar(model, EPOCHS)

## 8. Curvas de aprendizado (acurácia e erro)

Mostramos **treino × validação** para acurácia e perda (*loss*, o "erro"). A linha tracejada marca a época
com a **menor `val_loss`**, cujos pesos o `EarlyStopping` restaurou.

**Como ler:** o ideal é treino e validação andarem juntos. Sinais de **overfitting**: a acurácia de treino continua
subindo enquanto a de validação estaciona, e a `val_loss` para de cair e começa a subir. Como o treino usa
augmentation e dropout, é normal a acurácia de treino ficar *abaixo* da de validação nas primeiras épocas
(essas camadas só atuam no treino).

In [ ]:
def plotar_curvas(history, titulo=""):
    h = history.history
    epocas = range(1, len(h["loss"]) + 1)
    melhor = int(np.argmin(h["val_loss"])) + 1
    fig, ax = plt.subplots(1, 2, figsize=(14, 4))

    ax[0].plot(epocas, h["accuracy"], marker="o", label="treino")
    ax[0].plot(epocas, h["val_accuracy"], marker="o", label="validacao")
    ax[0].axvline(melhor, ls="--", color="gray", label=f"melhor epoca ({melhor})")
    ax[0].set_title("Acuracia " + titulo); ax[0].set_xlabel("Epoca"); ax[0].legend()

    ax[1].plot(epocas, h["loss"], marker="o", label="treino")
    ax[1].plot(epocas, h["val_loss"], marker="o", label="validacao")
    ax[1].axvline(melhor, ls="--", color="gray", label=f"melhor epoca ({melhor})")
    ax[1].set_title("Perda (loss) " + titulo); ax[1].set_xlabel("Epoca"); ax[1].legend()

    plt.tight_layout(); plt.show()

plotar_curvas(history, "- CNN base")

## 9. Avaliação no conjunto de teste

Agora usamos o conjunto de teste (que a rede nunca viu). Além da acurácia, calculamos:

- **Precision (precisão):** das imagens que a rede disse ser da classe X, quantas realmente eram? (evita falsos positivos)
- **Recall (revocação):** de todas as imagens que eram da classe X, quantas a rede encontrou? (evita falsos negativos)
- **F1-Score:** média harmônica de Precision e Recall; resume os dois em um número.
- **Média macro:** média simples entre as classes, sem ponderar pelo tamanho. É a mais sensível ao desempenho nas classes menores (importante em um dataset levemente desbalanceado).
- **Matriz de confusão:** mostra *onde* a rede erra (linhas = classe real, colunas = classe prevista).

In [ ]:
test_loss, test_acc = model.evaluate(test_ds, verbose=0)
print(f"Perda no teste: {test_loss:.4f}  |  Acuracia no teste: {test_acc * 100:.2f}%\n")

probs = model.predict(test_ds, verbose=0)    # (n_teste, 10) probabilidades
y_pred = probs.argmax(axis=1)                # classe prevista = indice da maior probabilidade

print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 8))
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(ax=ax, cmap="Blues", xticks_rotation=45)
ax.set_title("Matriz de confusao - conjunto de teste")
plt.tight_layout(); plt.show()

## 10. Imagens do teste: predição × real

Título **verde** = acerto, **vermelho** = erro. O percentual é a **confiança** (probabilidade da classe prevista).
Primeiro, 12 imagens sorteadas; depois, os erros da rede, para entender o que a confunde.

In [ ]:
# Junta todas as imagens do teste em um array (cabem na memoria: sao uint8 de 64x64x3)
x_test_imgs = np.concatenate([x.numpy() for x, _ in test_ds])

def mostrar_grade(indices, titulo, linhas, colunas):
    fig, axes = plt.subplots(linhas, colunas, figsize=(3.2 * colunas, 3.6 * linhas))
    for ax in np.ravel(axes):
        ax.axis("off")
    for ax, i in zip(np.ravel(axes), indices):
        ax.imshow(x_test_imgs[i])
        acertou = y_pred[i] == y_test[i]
        ax.set_title(f"previsto: {class_names[y_pred[i]]} ({probs[i].max() * 100:.0f}%)\n"
                     f"real: {class_names[y_test[i]]}",
                     color="green" if acertou else "red", fontsize=9)
    fig.suptitle(titulo, fontsize=13)
    plt.tight_layout(); plt.show()

rng = np.random.default_rng(SEED)
sorteadas = rng.choice(len(y_test), size=12, replace=False)
mostrar_grade(sorteadas, "Amostra aleatoria do conjunto de teste", 3, 4)

In [ ]:
erros = np.where(y_pred != y_test)[0]
print(f"A rede errou {len(erros)} de {len(y_test)} imagens de teste.")
if len(erros) > 0:
    mostrar_grade(erros[:8], "Alguns erros da rede", 2, 4)

## 11. Comparação de topologias

Treinamos **a mesma pipeline, com a mesma semente e os mesmos hiperparâmetros**, mudando apenas a topologia
(número de blocos convolucionais e de filtros) e, ao final, a presença de BatchNorm e de Dropout.
Assim, a única diferença entre os modelos é a que queremos medir.

| Modelo | Filtros por bloco | O que muda |
| --- | --- | --- |
| A | (16, 32) | rede **menor e mais rasa** (2 blocos) |
| B | (32, 64, 128) | **base** (3 blocos) |
| C | (32, 64, 128, 256) | **mais profunda** (4 blocos) |
| D | (64, 128, 256) | **mais larga** (dobro de filtros em 3 blocos) |
| E | (32, 64, 128), sem BatchNorm | efeito da **BatchNorm** |
| F | (32, 64, 128), sem Dropout | efeito do **Dropout** |

Comparamos pela **acurácia de validação** e, no teste, pelo **F1 macro**. Como esta seção é só
uma comparação, não usamos o teste para escolher hiperparâmetros durante o treino.

In [ ]:
EPOCHS_CMP = 15   # menos epocas que o treino principal, para a comparacao ser mais rapida

configs = {
    "A (16,32)":              dict(filters=(16, 32)),
    "B (32,64,128) base":     dict(filters=(32, 64, 128)),
    "C (32,64,128,256)":      dict(filters=(32, 64, 128, 256)),
    "D (64,128,256)":         dict(filters=(64, 128, 256)),
    "E base sem BatchNorm":   dict(filters=(32, 64, 128), batch_norm=False),
    "F base sem Dropout":     dict(filters=(32, 64, 128), dropout=0.0),
}

def avaliar(modelo):
    p = modelo.predict(test_ds, verbose=0)
    pred = p.argmax(axis=1)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, pred, average="macro", zero_division=0)
    return accuracy_score(y_test, pred), prec, rec, f1

resultados, historicos = [], {}
for i, (nome, cfg) in enumerate(configs.items()):
    print(f"\n=== Treinando {nome} ===")
    set_random_seed(SEED)
    m = build_cnn(name=f"cnn_{i}", **cfg)
    compilar(m)
    t0 = time.time()
    h = treinar(m, EPOCHS_CMP, verbose=0)
    duracao = time.time() - t0
    acc, prec, rec, f1 = avaliar(m)
    historicos[nome] = h
    resultados.append({
        "Modelo": nome,
        "Parametros": m.count_params(),
        "Epocas treinadas": len(h.history["loss"]),
        "Acc treino": max(h.history["accuracy"]),
        "Acc validacao": max(h.history["val_accuracy"]),
        "Acc teste": acc,
        "Precision (macro)": prec,
        "Recall (macro)": rec,
        "F1 (macro)": f1,
        "Tempo (s)": round(duracao),
    })
    print(f"acc validacao = {resultados[-1]['Acc validacao']:.3f} | acc teste = {acc:.3f} | F1 macro = {f1:.3f} | {duracao:.0f}s")

tabela = pd.DataFrame(resultados).set_index("Modelo")
tabela.style.format({"Parametros": "{:,}", "Acc treino": "{:.3f}", "Acc validacao": "{:.3f}", "Acc teste": "{:.3f}",
                     "Precision (macro)": "{:.3f}", "Recall (macro)": "{:.3f}", "F1 (macro)": "{:.3f}"})

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.5))

# Curvas de acuracia de validacao de todos os modelos
for nome, h in historicos.items():
    ax[0].plot(range(1, len(h.history["val_accuracy"]) + 1), h.history["val_accuracy"], marker=".", label=nome)
ax[0].set_title("Acuracia de validacao por topologia")
ax[0].set_xlabel("Epoca"); ax[0].set_ylabel("Acuracia (validacao)")
ax[0].legend(fontsize=8); ax[0].grid(True, alpha=0.3)

# F1 macro no teste
barras = ax[1].bar(range(len(tabela)), tabela["F1 (macro)"], color="tab:blue")
ax[1].bar_label(barras, fmt="%.3f")
ax[1].set_xticks(range(len(tabela)))
ax[1].set_xticklabels([n.split(" ")[0] for n in tabela.index])
ax[1].set_ylim(max(0, tabela["F1 (macro)"].min() - 0.1), 1)
ax[1].set_title("F1-Score macro no teste")
ax[1].set_xlabel("Modelo (A a F)")

plt.tight_layout(); plt.show()

## 12. Conclusões

> ✍️ **Complete esta seção com os números e gráficos que você obteve** (os resultados mudam um pouco
> conforme a GPU e a versão do TensorFlow). Sugestão de roteiro:

**Desempenho da CNN base**
- Acurácia / F1 macro no teste: *…*. A rede generalizou bem? Compare treino × validação nas curvas (há overfitting? em qual época a `val_loss` foi mínima?).
- Quais classes tiveram melhor e pior F1? Consulte a matriz de confusão: quais pares são mais confundidos (por exemplo, *Highway* × *River*, ou as classes de vegetação e cultivo entre si: *AnnualCrop*, *PermanentCrop*, *Pasture*, *HerbaceousVegetation*) e por quê (textura, cor e formato parecidos vistos de cima)?
- Os erros mostrados na seção 10 têm algo em comum (recortes com mais de um tipo de cobertura, imagens de baixa resolução, classes ambíguas)?

**Comparação de topologias (seção 11)**
- Aumentar a profundidade (C) ou a largura (D) melhorou o resultado em relação à base (B)? Compensou o aumento de parâmetros e de tempo?
- A rede menor (A) ficou muito pior? Isso indica *underfitting*?
- O que mudou ao remover BatchNorm (E) e Dropout (F)? Veja a velocidade de convergência e a distância entre treino e validação.

**Limitações e próximos passos**
- Imagens de baixa resolução (64×64) e de regiões da Europa, então o modelo pode não generalizar para outros contextos (por exemplo, o Brasil); a versão RGB não usa bandas infravermelhas.
- Ideias: usar as 13 bandas espectrais (EuroSAT_MS), `class_weight` para o leve desbalanceamento, mais épocas, *transfer learning* (por exemplo, MobileNetV2 pré-treinada no ImageNet) e validação cruzada.